In [2]:
from pathlib import Path

APP_DIR = Path("../app")
APP_DIR.mkdir(parents=True, exist_ok=True)

print("✅ App directory ready")


✅ App directory ready


In [3]:
streamlit_code = 'import requests\nimport pandas as pd\nimport streamlit as st\nimport folium\nfrom streamlit_folium import st_folium\n\n\nAPI_BASE_URL = "http://127.0.0.1:8000"\n\n\nst.set_page_config(\n    page_title="TravelMate AI",\n    page_icon="🌍",\n    layout="wide",\n)\n\n\n# ---------------------------------------------------------\n# Session state\n# ---------------------------------------------------------\n\nif "trip_result" not in st.session_state:\n    st.session_state.trip_result = None\n\nif "api_error" not in st.session_state:\n    st.session_state.api_error = None\n\n\n# ---------------------------------------------------------\n# Helper functions\n# ---------------------------------------------------------\n\n@st.cache_data(ttl=300)\ndef get_supported_cities():\n    response = requests.get(\n        f"{API_BASE_URL}/cities",\n        timeout=20,\n    )\n    response.raise_for_status()\n\n    return response.json().get("cities", [])\n\n\ndef generate_trip(\n    destination,\n    query,\n    days,\n    top_n,\n    preferences,\n):\n    payload = {\n        "destination": destination,\n        "query": query,\n        "days": int(days),\n        "top_n": int(top_n),\n        "preferences": preferences,\n    }\n\n    recommendation_response = requests.post(\n        f"{API_BASE_URL}/recommend",\n        json=payload,\n        timeout=120,\n    )\n    recommendation_response.raise_for_status()\n\n    itinerary_response = requests.post(\n        f"{API_BASE_URL}/itinerary",\n        json=payload,\n        timeout=120,\n    )\n    itinerary_response.raise_for_status()\n\n    return {\n        "destination": destination,\n        "query": query,\n        "recommendations":\n            recommendation_response.json().get(\n                "recommendations",\n                [],\n            ),\n        "itinerary":\n            itinerary_response.json().get(\n                "itinerary",\n                [],\n            ),\n    }\n\n\ndef format_rating(rating):\n    try:\n        return f"{float(rating):.1f}"\n    except (TypeError, ValueError):\n        return "N/A"\n\n\n# ---------------------------------------------------------\n# Sidebar\n# ---------------------------------------------------------\n\nst.sidebar.header("🧳 Trip Preferences")\n\n\ntry:\n    cities = get_supported_cities()\nexcept Exception as exc:\n    cities = []\n    st.sidebar.error(\n        "❌ FastAPI is not available. "\n        "Start it with: uvicorn app.main:app --reload"\n    )\n\n\nif cities:\n    default_index = (\n        cities.index("Manali")\n        if "Manali" in cities\n        else 0\n    )\n\n    destination = st.sidebar.selectbox(\n        "Destination",\n        cities,\n        index=default_index,\n    )\nelse:\n    destination = st.sidebar.text_input(\n        "Destination",\n        value="Manali",\n    )\n\n\ndays = st.sidebar.number_input(\n    "Number of days",\n    min_value=1,\n    max_value=14,\n    value=3,\n    step=1,\n)\n\ntop_n = st.sidebar.slider(\n    "Number of recommendations",\n    min_value=3,\n    max_value=15,\n    value=6,\n)\n\nst.sidebar.markdown("### ❤️ What do you like?")\n\nnature = st.sidebar.slider(\n    "🌿 Nature",\n    0.0, 1.0, 0.8, 0.1\n)\n\nhistory = st.sidebar.slider(\n    "🏛️ History",\n    0.0, 1.0, 0.2, 0.1\n)\n\nculture = st.sidebar.slider(\n    "🎭 Culture",\n    0.0, 1.0, 0.3, 0.1\n)\n\nadventure = st.sidebar.slider(\n    "🥾 Adventure",\n    0.0, 1.0, 0.4, 0.1\n)\n\nphotography = st.sidebar.slider(\n    "📸 Photography",\n    0.0, 1.0, 0.8, 0.1\n)\n\nshopping = st.sidebar.slider(\n    "🛍️ Shopping",\n    0.0, 1.0, 0.2, 0.1\n)\n\nreligious = st.sidebar.slider(\n    "🛕 Religious",\n    0.0, 1.0, 0.1, 0.1\n)\n\nfamily = st.sidebar.slider(\n    "👨\u200d👩\u200d👧 Family",\n    0.0, 1.0, 0.4, 0.1\n)\n\n\npreferences = {\n    "nature": nature,\n    "history": history,\n    "culture": culture,\n    "adventure": adventure,\n    "photography": photography,\n    "shopping": shopping,\n    "religious": religious,\n    "family": family,\n}\n\n\n# ---------------------------------------------------------\n# Main page\n# ---------------------------------------------------------\n\nst.title("🌍 TravelMate AI")\n\nst.subheader(\n    "Your personalized AI travel planner ✈️"\n)\n\nst.write(\n    "Choose a destination, describe your trip, "\n    "and TravelMate will recommend places and build "\n    "a day-by-day itinerary."\n)\n\n\nquery = st.text_area(\n    "💬 Describe your trip",\n    value=(\n        "I want a peaceful scenic trip "\n        "with beautiful places for photography."\n    ),\n    height=110,\n)\n\n\ngenerate = st.button(\n    "✨ Generate My Trip",\n    type="primary",\n    width="stretch",\n)\n\n\nif generate:\n\n    try:\n        with st.spinner(\n            "🤖 TravelMate is creating your trip..."\n        ):\n\n            st.session_state.trip_result = (\n                generate_trip(\n                    destination=destination,\n                    query=query,\n                    days=days,\n                    top_n=top_n,\n                    preferences=preferences,\n                )\n            )\n\n        st.session_state.api_error = None\n\n    except requests.exceptions.ConnectionError:\n\n        st.session_state.trip_result = None\n\n        st.session_state.api_error = (\n            "❌ Cannot connect to FastAPI. "\n            "Run `uvicorn app.main:app --reload` first."\n        )\n\n    except requests.exceptions.Timeout:\n\n        st.session_state.trip_result = None\n\n        st.session_state.api_error = (\n            "⏳ The AI backend took too long to respond."\n        )\n\n    except requests.exceptions.HTTPError as exc:\n\n        st.session_state.trip_result = None\n\n        detail = str(exc)\n\n        try:\n            detail = exc.response.json().get(\n                "detail",\n                detail,\n            )\n        except Exception:\n            pass\n\n        st.session_state.api_error = (\n            f"❌ API error: {detail}"\n        )\n\n    except Exception as exc:\n\n        st.session_state.trip_result = None\n\n        st.session_state.api_error = (\n            f"❌ Unexpected error: {exc}"\n        )\n\n\n# ---------------------------------------------------------\n# Error display\n# ---------------------------------------------------------\n\nif st.session_state.api_error:\n    st.error(\n        st.session_state.api_error\n    )\n\n\n# ---------------------------------------------------------\n# Results\n# ---------------------------------------------------------\n\nresult = st.session_state.trip_result\n\nif result:\n\n    st.success(\n        f"✅ Trip generated for "\n        f"{result[\'destination\']}"\n    )\n\n    recommendations = result[\n        "recommendations"\n    ]\n\n    itinerary = result["itinerary"]\n\n\n    # -----------------------------------------------------\n    # Recommendations\n    # -----------------------------------------------------\n\n    st.header("🎯 Recommended Places")\n\n    if recommendations:\n\n        rec_df = pd.DataFrame(\n            recommendations\n        )\n\n        for column in [\n            "final_score",\n            "semantic_score",\n            "tfidf_score",\n            "structured_score",\n        ]:\n            if column in rec_df.columns:\n                rec_df[column] = rec_df[column].round(4)\n\n        display_columns = [\n            "name",\n            "activity_type",\n            "rating",\n            "reviews",\n            "estimated_visit_minutes",\n            "final_score",\n        ]\n\n        available = [\n            column\n            for column in display_columns\n            if column in rec_df.columns\n        ]\n\n        st.dataframe(\n            rec_df[available],\n            width="stretch",\n            hide_index=True,\n        )\n\n        # Top place cards\n        st.subheader("⭐ Top Picks")\n\n        card_columns = st.columns(\n            min(3, len(rec_df))\n        )\n\n        for column, (_, row) in zip(\n            card_columns,\n            rec_df.head(3).iterrows(),\n        ):\n\n            with column:\n\n                st.markdown(\n                    f"### 📍 {row[\'name\']}"\n                )\n\n                st.write(\n                    f"⭐ Rating: "\n                    f"{format_rating(row.get(\'rating\'))}"\n                )\n\n                st.write(\n                    f"📝 Reviews: "\n                    f"{int(row.get(\'reviews\', 0)):,}"\n                )\n\n                if "activity_type" in row:\n                    st.write(\n                        f"🎯 Activity: "\n                        f"{row[\'activity_type\']}"\n                    )\n\n                if "final_score" in row:\n                    st.write(\n                        f"🤖 AI Score: "\n                        f"{float(row[\'final_score\']):.3f}"\n                    )\n\n    else:\n\n        st.warning(\n            "No recommendations were returned."\n        )\n\n\n    # -----------------------------------------------------\n    # Itinerary\n    # -----------------------------------------------------\n\n    st.header("📅 Your Personalized Itinerary")\n\n    if itinerary:\n\n        itinerary_df = pd.DataFrame(\n            itinerary\n        )\n\n        # Summary metrics\n        total_visit_minutes = int(\n            itinerary_df[\n                "visit_minutes"\n            ].sum()\n        )\n\n        total_travel_minutes = float(\n            itinerary_df[\n                "travel_before_minutes"\n            ].sum()\n        )\n\n        scheduled_stops = len(\n            itinerary_df\n        )\n\n        metric_cols = st.columns(3)\n\n        metric_cols[0].metric(\n            "📍 Stops",\n            scheduled_stops,\n        )\n\n        metric_cols[1].metric(\n            "⏱️ Visit Time",\n            f"{total_visit_minutes} min",\n        )\n\n        metric_cols[2].metric(\n            "🚗 Travel Time",\n            f"{total_travel_minutes:.1f} min",\n        )\n\n\n        for day in sorted(\n            itinerary_df["day"].unique()\n        ):\n\n            day_df = (\n                itinerary_df[\n                    itinerary_df["day"] == day\n                ]\n                .sort_values("stop")\n            )\n\n            st.subheader(\n                f"📍 Day {int(day)}"\n            )\n\n            for _, row in day_df.iterrows():\n\n                with st.container(\n                    border=True\n                ):\n\n                    st.markdown(\n                        f"### Stop {int(row[\'stop\'])} "\n                        f"• {row[\'place\']}"\n                    )\n\n                    left, right = st.columns(2)\n\n                    with left:\n                        st.write(\n                            f"🕘 "\n                            f"{row[\'arrival\']} → "\n                            f"{row[\'departure\']}"\n                        )\n\n                        st.write(\n                            f"🎯 "\n                            f"{row[\'activity_type\']}"\n                        )\n\n                    with right:\n                        st.write(\n                            f"⏱️ Visit: "\n                            f"{int(row[\'visit_minutes\'])} min"\n                        )\n\n                        st.write(\n                            f"🚗 Travel before: "\n                            f"{float(row[\'travel_before_minutes\']):.1f} min"\n                        )\n\n            st.divider()\n\n    else:\n\n        st.warning(\n            "No itinerary could be generated."\n        )\n\n\n    # -----------------------------------------------------\n    # Map\n    # -----------------------------------------------------\n\n    st.header("🗺️ Interactive Trip Map")\n\n    map_points = [\n        item\n        for item in itinerary\n        if item.get("latitude") is not None\n        and item.get("longitude") is not None\n    ]\n\n    if map_points:\n\n        center_lat = sum(\n            item["latitude"]\n            for item in map_points\n        ) / len(map_points)\n\n        center_lon = sum(\n            item["longitude"]\n            for item in map_points\n        ) / len(map_points)\n\n        travel_map = folium.Map(\n            location=[\n                center_lat,\n                center_lon,\n            ],\n            zoom_start=12,\n        )\n\n        for item in map_points:\n\n            popup_html = (\n                f"<b>Day {item[\'day\']} "\n                f"• Stop {item[\'stop\']}</b><br>"\n                f"{item[\'place\']}<br>"\n                f"{item[\'arrival\']} - "\n                f"{item[\'departure\']}"\n            )\n\n            folium.Marker(\n                location=[\n                    item["latitude"],\n                    item["longitude"],\n                ],\n                popup=popup_html,\n                tooltip=item["place"],\n            ).add_to(travel_map)\n\n        # Draw a separate line for each day.\n        for day in sorted(\n            {\n                item["day"]\n                for item in map_points\n            }\n        ):\n\n            day_points = sorted(\n                [\n                    item\n                    for item in map_points\n                    if item["day"] == day\n                ],\n                key=lambda item: item["stop"],\n            )\n\n            coordinates = [\n                [\n                    item["latitude"],\n                    item["longitude"],\n                ]\n                for item in day_points\n            ]\n\n            if len(coordinates) >= 2:\n\n                folium.PolyLine(\n                    coordinates,\n                    tooltip=f"Day {day} route",\n                ).add_to(travel_map)\n\n        st_folium(\n            travel_map,\n            height=600,\n            width=None,\n        )\n\n    else:\n\n        st.info(\n            "Map coordinates are unavailable "\n            "for this itinerary."\n        )\n\n\nelse:\n\n    st.info(\n        "👈 Choose your destination and "\n        "preferences, then click "\n        "**Generate My Trip**."\n    )\n\n\nst.caption(\n    "TravelMate AI • Hybrid Recommendation "\n    "• Itinerary Optimization • Interactive Map"\n)\n'
app_path = APP_DIR / "streamlit_app.py"

app_path.write_text(
    streamlit_code,
    encoding="utf-8"
)

print(f"✅ Final Streamlit app created: {app_path}")


✅ Final Streamlit app created: ..\app\streamlit_app.py


## 3. Verify the generated app

In [4]:
app_path = APP_DIR / "streamlit_app.py"

print(
    "Exists:",
    app_path.exists()
)

print(
    "Size:",
    app_path.stat().st_size,
    "bytes"
)

print(
    "\nFirst 15 lines:"
)

print(
    "\n".join(
        app_path.read_text(
            encoding="utf-8"
        ).splitlines()[:15]
    )
)


Exists: True
Size: 14467 bytes

First 15 lines:
import requests
import pandas as pd
import streamlit as st
import folium
from streamlit_folium import st_folium


API_BASE_URL = "http://127.0.0.1:8000"


st.set_page_config(
    page_title="TravelMate AI",
    page_icon="🌍",
    layout="wide",
)
